# Kinematic Guardrail T2.5 + T2.6 Validation\n
\n
This notebook validates:\n
- T2.5 rigid topology preservation (SSIM-based)\n
- T2.6 composite L_physics = L_bone + L_ROM + L_velocity\n
\n
Scenarios include rigid translation, localized deformation, and high-velocity motion.

In [ ]:
from pathlib import Path
import sys
import inspect

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'core').exists() is False:
    PROJECT_ROOT = Path.cwd().resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from core import kinematic_guardrail as kg

compute_bone_lengths = kg.compute_bone_lengths
bone_length_invariance_loss = kg.bone_length_invariance_loss
compute_velocity_loss = kg.compute_velocity_loss
compute_rigid_topology_loss = getattr(kg, "compute_rigid_topology_loss", None)

if compute_rigid_topology_loss is None:
    def compute_rigid_topology_loss(pose_keypoints, *args, **kwargs):
        # Backward-compat fallback when T2.5 is unavailable in remote environment.
        pose_keypoints = np.asarray(pose_keypoints, dtype=np.float32)
        if pose_keypoints.ndim == 3:
            pose_keypoints = pose_keypoints[None, ...]
        b = pose_keypoints.shape[0]
        t = max(pose_keypoints.shape[1] - 1, 0)
        return {"head": np.ones((b, t), dtype=np.float32), "torso": np.ones((b, t), dtype=np.float32), "pelvis": np.ones((b, t), dtype=np.float32)}, 0.0

if hasattr(kg, "compute_rom_loss"):
    compute_rom_loss = kg.compute_rom_loss
else:
    JOINT_ANGLE_LIMITS = {
        5: (0.0, 180.0),
        6: (0.0, 180.0),
        7: (0.0, 145.0),
        8: (0.0, 145.0),
        11: (0.0, 120.0),
        12: (0.0, 120.0),
        13: (0.0, 150.0),
        14: (0.0, 150.0),
    }
    JOINT_ANGLE_TRIPLETS = {
        5: (7, 5, 11),
        6: (8, 6, 12),
        7: (9, 7, 5),
        8: (10, 8, 6),
        11: (13, 11, 5),
        12: (14, 12, 6),
        13: (15, 13, 11),
        14: (16, 14, 12),
    }

    def _angle_degrees(a, b, c):
        ba = a - b
        bc = c - b
        dot = np.sum(ba * bc, axis=-1)
        denom = np.maximum(np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1), 1e-8)
        cos_theta = np.clip(dot / denom, -1.0, 1.0)
        return np.degrees(np.arccos(cos_theta)).astype(np.float32)

    def compute_rom_loss(pose_keypoints, joint_limits=None):
        # Local fallback to keep notebook runnable on older commits.
        points = np.asarray(pose_keypoints, dtype=np.float32)
        if points.ndim == 3:
            points = points[None, ...]
        if points.ndim != 4 or points.shape[-1] < 2:
            raise ValueError("pose_keypoints must have shape [B, T, K, 2] or [T, K, 2].")
        points = points[..., :2]
        limits = JOINT_ANGLE_LIMITS if joint_limits is None else dict(joint_limits)
        ordered_joints = sorted(limits.keys())
        angles = np.zeros((points.shape[0], points.shape[1], len(ordered_joints)), dtype=np.float32)
        for j, joint_idx in enumerate(ordered_joints):
            a_idx, b_idx, c_idx = JOINT_ANGLE_TRIPLETS[joint_idx]
            angles[:, :, j] = _angle_degrees(points[:, :, a_idx, :], points[:, :, b_idx, :], points[:, :, c_idx, :])
        penalties = np.zeros_like(angles, dtype=np.float32)
        for j, joint_idx in enumerate(ordered_joints):
            min_angle, max_angle = limits[joint_idx]
            lower = np.maximum(0.0, float(min_angle) - angles[:, :, j])
            upper = np.maximum(0.0, angles[:, :, j] - float(max_angle))
            penalties[:, :, j] = (lower + upper) / 180.0
        return angles, float(np.sum(np.square(penalties), dtype=np.float64))

def compute_l_physics(pose_keypoints, bone_weight=1.0, rom_weight=1.0, velocity_weight=1.0):
    # Notebook-stable T2.6 definition independent of core module version.
    _, bone_loss = bone_length_invariance_loss(pose_keypoints)
    _, rom_loss = compute_rom_loss(pose_keypoints)
    _, velocity_loss = compute_velocity_loss(pose_keypoints, v_max=2.0)
    _, topology_loss = compute_rigid_topology_loss(pose_keypoints)
    total_loss = bone_weight * bone_loss + rom_weight * rom_loss + velocity_weight * velocity_loss
    return {
        "bone_loss": float(bone_loss),
        "rom_loss": float(rom_loss),
        "velocity_loss": float(velocity_loss),
        "topology_loss": float(topology_loss),
        "total_loss": float(total_loss),
    }

np.set_printoptions(precision=4, suppress=True)
print("Imports complete. Project root:", PROJECT_ROOT)
print("compute_rom_loss source:", "core" if hasattr(kg, "compute_rom_loss") else "notebook fallback")
print("compute_l_physics source: notebook T2.6 wrapper")

In [ ]:
def make_human_like_pose():\n
    return np.array([\n
        [0.00, 1.90],   # 0 nose\n
        [-0.10, 2.00],  # 1 left_eye\n
        [0.10, 2.00],   # 2 right_eye\n
        [-0.22, 1.98],  # 3 left_ear\n
        [0.22, 1.98],   # 4 right_ear\n
        [-0.30, 1.55],  # 5 left_shoulder\n
        [0.30, 1.55],   # 6 right_shoulder\n
        [-0.48, 1.20],  # 7 left_elbow\n
        [0.48, 1.20],   # 8 right_elbow\n
        [-0.58, 0.90],  # 9 left_wrist\n
        [0.58, 0.90],   # 10 right_wrist\n
        [-0.20, 1.00],  # 11 left_hip\n
        [0.20, 1.00],   # 12 right_hip\n
        [-0.20, 0.55],  # 13 left_knee\n
        [0.20, 0.55],   # 14 right_knee\n
        [-0.20, 0.10],  # 15 left_ankle\n
        [0.20, 0.10],   # 16 right_ankle\n
    ], dtype=np.float32)\n
\n
def as_sequence(frames):\n
    # Convert list of [K,2] frames to [B,T,K,2] with B=1\n
    return np.stack(frames, axis=0)[None, ...]\n
\n
base = make_human_like_pose()\n
print('Base pose shape:', base.shape)

In [ ]:
# Scenario A: rigid translation (expected low L_bone and low topology loss)\n
frames_a = [\n
    base + np.array([0.00, 0.00], dtype=np.float32),\n
    base + np.array([0.30, -0.20], dtype=np.float32),\n
    base + np.array([0.60, -0.40], dtype=np.float32),\n
]\n
seq_a = as_sequence(frames_a)\n
\n
_, l_bone_a = bone_length_invariance_loss(seq_a)\n
_, l_rom_a = compute_rom_loss(seq_a)\n
_, l_vel_a = compute_velocity_loss(seq_a, v_max=2.0)\n
region_ssim_a, l_topo_a = compute_rigid_topology_loss(seq_a)\n
l_phys_a = compute_l_physics(seq_a, bone_weight=1.0, rom_weight=1.0, velocity_weight=1.0)\n
\n
print('Scenario A - Rigid Translation')\n
print('L_bone      :', l_bone_a)\n
print('L_ROM       :', l_rom_a)\n
print('L_velocity  :', l_vel_a)\n
print('L_topology  :', l_topo_a)\n
print('L_physics   :', l_phys_a['total_loss'])\n
print('Region SSIM means:', {k: float(v.mean()) for k, v in region_ssim_a.items()})

In [ ]:
# Scenario B: local deformation (expected higher L_bone / L_topology / possibly L_ROM)\n
deformed = base.copy()\n
deformed[6] += np.array([0.55, -0.35], dtype=np.float32)  # right shoulder shift\n
deformed[15] += np.array([1.10, 0.00], dtype=np.float32) # left ankle shift (ROM stress)\n
\n
seq_b = as_sequence([base, deformed])\n
\n
_, l_bone_b = bone_length_invariance_loss(seq_b)\n
_, l_rom_b = compute_rom_loss(seq_b)\n
_, l_vel_b = compute_velocity_loss(seq_b, v_max=2.0)\n
region_ssim_b, l_topo_b = compute_rigid_topology_loss(seq_b)\n
l_phys_b = compute_l_physics(seq_b, bone_weight=1.0, rom_weight=1.0, velocity_weight=1.0)\n
\n
print('Scenario B - Local Deformation')\n
print('L_bone      :', l_bone_b)\n
print('L_ROM       :', l_rom_b)\n
print('L_velocity  :', l_vel_b)\n
print('L_topology  :', l_topo_b)\n
print('L_physics   :', l_phys_b['total_loss'])\n
print('Region SSIM means:', {k: float(v.mean()) for k, v in region_ssim_b.items()})

In [ ]:
# Scenario C: high velocity motion (expected high L_velocity)\n
fast_0 = base\n
fast_1 = base + np.array([3.5, 0.0], dtype=np.float32)\n
seq_c = as_sequence([fast_0, fast_1])\n
\n
velocities_c, l_vel_c = compute_velocity_loss(seq_c, v_max=2.0)\n
l_phys_c = compute_l_physics(seq_c, bone_weight=1.0, rom_weight=1.0, velocity_weight=1.0)\n
\n
print('Scenario C - High Velocity')\n
print('Mean velocity:', float(velocities_c.mean()))\n
print('L_velocity   :', l_vel_c)\n
print('L_physics    :', l_phys_c['total_loss'])

In [ ]:
# Compare weighted composite losses for scenario B\n
weights = [\n
    {'bone_weight': 1.0, 'rom_weight': 1.0, 'velocity_weight': 1.0},\n
    {'bone_weight': 2.0, 'rom_weight': 0.5, 'velocity_weight': 1.0},\n
    {'bone_weight': 0.5, 'rom_weight': 2.0, 'velocity_weight': 1.0},\n
]\n
\n
results = []\n
for w in weights:\n
    out = compute_l_physics(seq_b, **w)\n
    results.append((w, out))\n
\n
for w, out in results:\n
    print('Weights:', w)\n
    print('  bone_loss   =', out['bone_loss'])\n
    print('  rom_loss    =', out['rom_loss'])\n
    print('  velocity    =', out['velocity_loss'])\n
    print('  topology    =', out['topology_loss'])\n
    print('  total_loss  =', out['total_loss'])

In [ ]:
# Optional quick visual: base vs deformed keypoints\n
fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n
\n
axes[0].scatter(base[:, 0], base[:, 1], c='green')\n
axes[0].set_title('Base Pose')\n
axes[0].invert_yaxis()\n
axes[0].set_aspect('equal', adjustable='box')\n
\n
axes[1].scatter(deformed[:, 0], deformed[:, 1], c='red')\n
axes[1].set_title('Deformed Pose')\n
axes[1].invert_yaxis()\n
axes[1].set_aspect('equal', adjustable='box')\n
\n
for ax in axes:\n
    ax.grid(True, alpha=0.3)\n
\n
plt.tight_layout()\n
plt.show()